In [ ]:
import torch
import math

from src.knots import *
from src.extensions import *
from src.full_models import HyperbolicMinimalSurfacePINN
from src.plotting import plot_error, plot_knot, montecarlo_error
from src.geometry import minimal_in_H4_PDE_flat_new
from src.samplers import MixSampler
from src.training import train_PINN_Adam, refine_PINN_lbfgs
from src.interior_models import MLP

from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget

In [ ]:
# ------------------
# Model construction
# ------------------
model_kwargs = {
    # 1. knot
    'knot_type': 'stevedore',
    'knot_kwargs': {
        'R': 1.6,
        'mirror': True},
    'knot_perturbation_matrix': generate_knot_perturbation_matrix(
        N=4,
        scale=0.1,
        seed=11,),
    # 'knot_perturbation_matrix': mat,

    # 2. interior model
    'interior_model_type': 'mlp',
    'interior_model_kwargs': {
        'in_dim': 2,
        'out_dim': 4,
        'hidden_dim': 64,
        'num_hidden_layers': 4
    },

    # 3. bdf
    'bdf_type': 'stereographic',

    # 4. extension
    'ext_type': 'stereobiharmonic',
    'ext_kwargs': {'N': 15, 'num_samples': 10000},

    # 5. decay exponent
    'decay_exponent': 2
}

model = HyperbolicMinimalSurfacePINN(**model_kwargs)

In [ ]:
knot = model.get_knot()

plot_knot(knot.get_evaluator())

In [ ]:
fast_pde = minimal_in_H4_PDE_flat_new(
    model,
    use_compile=True, 
    compile_kwargs={"mode": "default", "fullgraph": True, "dynamic": False}
)

In [ ]:
plot_error(
    minimal_in_H4_PDE_flat_new(model),
    dtype = model.kwargs['dtype'],
    vmin = 0,
    vmax = 1,
    grid_size=200)

In [ ]:
sampler = MixSampler(dtype=model.kwargs['dtype'])

xy = sampler(2**14)

residuals = minimal_in_H4_PDE_flat_new(model)(xy)
MSE = (residuals**2).sum(dim=-1).mean()

MSE

In [ ]:
model.name

In [ ]:
n_points = 2**14
xy = sampler(n_points)

best_loss, _ = train_PINN_Adam(
    model,
    xy_grid=xy,              
    epochs=10000,
    batch_size=2**10,
    lr=1e-3,
    lr_min=1e-5,
    scheduler_type='cosine',  # Choose 'plateau' or 'cosine'
    # scheduler_patience=20,    # Used only if scheduler_type = 'plateau'
    # scheduler_factor=0.9,     # Used only if scheduler_type = 'plateau'
    # scheduler_threshold=1e-2, # Used only if scheduler_type = 'plateau'
    verbose=True
)

In [ ]:
plot_error(
    minimal_in_H4_PDE_flat_new(model),
    dtype = model.kwargs['dtype'],
    vmin = 0,
    vmax = 1,
    grid_size=200)

In [ ]:
xy = sampler(2**14)

_ = refine_PINN_lbfgs(
    model,
    xy_grid=xy,              
    lr=1.0,
    max_iter=100000,
    log_every=10,
    tolerance_grad=1e-10,
    tolerance_change=1e-12
)

In [ ]:
plot_error(
    minimal_in_H4_PDE_flat_new(model),
    dtype=model.kwargs['dtype'],
    vmin = 0,
    vmax = 1,
    grid_size=200)

In [ ]:
trained_models_path = 'trained_models'
model.save(directory=trained_models_path)